# MedAssist-AI: Disease Prediction using Random Forest

**Week 1 – Disease Prediction & Risk Assessment**  
This notebook demonstrates dataset loading, preprocessing, an 80:20 train-test split, Random Forest training, disease prediction, and model evaluation.

> Run this notebook from the project root (`MedAssist-AI`). The dataset path below matches the project file `Disease_symptom_and_patient_profile_dataset.csv`.

**Note:** This is an educational prediction workflow, not a medical diagnosis tool. The model's predictions should not be used to make healthcare decisions.

## 1. Import required libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib
from pathlib import Path

## 2. Load the Kaggle dataset

In [ ]:
DATA_PATH = Path("Disease_symptom_and_patient_profile_dataset.csv")

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()

## 3. Inspect and preprocess the dataset
Check missing values, remove duplicate rows, and prepare the data for training.

In [ ]:
print("Missing values before cleaning:")
display(df.isnull().sum())

print("Duplicate rows before cleaning:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df = df.dropna().copy()

print("Shape after cleaning:", df.shape)
print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Missing values after cleaning:")
display(df.isnull().sum())

## 4. Select features and target

The `Disease` column is the target. The remaining patient and symptom columns are used as input features. Categorical inputs are encoded using `OneHotEncoder` inside a Scikit-learn pipeline.

In [ ]:
target_column = "Disease"

if target_column not in df.columns:
    raise ValueError(f"Expected target column '{target_column}' was not found.")

X = df.drop(columns=[target_column])
y = df[target_column].astype(str)

print("Feature columns:", list(X.columns))
print("Number of disease classes before filtering:", y.nunique())

## 5. Prepare disease classes

To match the project's current training approach, retain disease classes with at least 10 records. This helps avoid training on classes represented by only a few examples. The final number of classes is displayed below.

In [ ]:
class_counts = y.value_counts()
valid_classes = class_counts[class_counts >= 10].index

filtered = df[df[target_column].astype(str).isin(valid_classes)].copy()
X = filtered.drop(columns=[target_column])
y = filtered[target_column].astype(str)

print("Shape after class filtering:", filtered.shape)
print("Disease classes:", sorted(y.unique()))
display(y.value_counts())

## 6. Encode the target and split the data (80:20)

Disease labels are encoded as integers for model training. The split is stratified when each retained class has enough samples to appear in both sets.

In [ ]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Use stratification so the train and test sets preserve class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 7. Build and train the Random Forest model

In [ ]:
categorical_columns = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("numeric", "passthrough", numeric_columns),
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
print("Random Forest model training completed.")

## 8. Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(len(label_encoder.classes_)),
    target_names=label_encoder.classes_,
    zero_division=0
))

## 9. Compare actual and predicted diseases

In [ ]:
results = pd.DataFrame({
    "Actual Disease": label_encoder.inverse_transform(y_test),
    "Predicted Disease": label_encoder.inverse_transform(y_pred)
})
results["Correct"] = results["Actual Disease"] == results["Predicted Disease"]
display(results.head(20))

## 10. Generate disease probability scores

In [ ]:
probabilities = model.predict_proba(X_test)

probability_results = pd.DataFrame(
    probabilities,
    columns=label_encoder.classes_
)
probability_results.insert(
    0,
    "Predicted Disease",
    label_encoder.inverse_transform(y_pred)
)
display(probability_results.head())

## 11. Save the trained model

The model and disease labels are saved under `backend/models` so the backend can reuse them. If the project already has model files, review before replacing them.

In [ ]:
models_dir = Path("backend") / "models"
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model, models_dir / "disease_prediction_pipeline.joblib")
joblib.dump(label_encoder, models_dir / "disease_label_encoder.joblib")

print("Saved model files to:", models_dir.resolve())

## 12. Conclusion

This notebook loads and cleans the Kaggle dataset, encodes input features and disease labels, trains a Random Forest classifier using an 80:20 train-test split, and evaluates predictions against the actual disease labels. It also displays class probabilities and saves the trained pipeline.

**Important:** The reported accuracy applies only to this dataset and split. The dataset is small and may not represent real patients; this model is for project demonstration only.